In [1]:
import importlib
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd

warnings.simplefilter("ignore")

PROJECT_ROOT = Path.cwd().resolve()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "analysis" / "src").exists() and (candidate / "data").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError(f"Could not locate project root from {Path.cwd().resolve()}")
DROPBOX_ROOT = Path("/Users/satoshi/Library/CloudStorage/Dropbox/HSA_NKPC_MCMC")
sys.path.append(str(PROJECT_ROOT / "analysis" / "src"))
sys.path.append(str(PROJECT_ROOT / "analysis" / "gibbs"))

DATA_DIR = PROJECT_ROOT / "data"
IDATA_ROOT = DROPBOX_ROOT / "results" / "idata"
TEX_ROOT_DIR = PROJECT_ROOT / "results" / "tex" / "table"
FIG_ROOT = PROJECT_ROOT / "results" / "fig"

# Change DROPBOX_ROOT above if your Dropbox location differs.
# IDATA files will be saved to and loaded from IDATA_ROOT.
from func_data_build import build_dataset
from func_gibbs.gibbs_notebook_utils import display_hmc_results, display_hmc_posterior_prior, save_idata_map
from func_gibbs.gibbs_wrappers import draws_to_idata
import func_gibbs.gibbs_wrappers as gibbs_module
from func_gibbs.gibbs_wrappers import run_hsa_steady

data_dir = DATA_DIR
idat_dir = IDATA_ROOT
tex_root_dir = TEX_ROOT_DIR
base_fig_dir = FIG_ROOT

## 0. load data
data = build_dataset(data_dir)
data["DATE"] = pd.to_datetime(data.index)

## make model-specific maximum available sample
def make_model_sample(data, spec, inflation_specs, include_N=True):
    infl = inflation_specs[spec["inflation"]]
    gap = spec["gap"]

    required_cols = [
        infl["pi"],
        infl["pi_prev"],
        infl["pi_expect"],
        gap,
        f"{gap}_prev",
    ]

    if include_N:
        required_cols.append(spec["N"])

    missing_cols = [col for col in required_cols if col not in data.columns]
    if missing_cols:
        raise KeyError(f"Missing columns for {spec['model_id']}: {missing_cols}")

    sample = (
        data[required_cols]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        .copy()
    )

    if sample.empty:
        raise ValueError(f"No valid sample for {spec['model_id']}")

    return sample


In [2]:
## model specifications
inflation_specs = {
    "ppi": {
        "pi": "pi_ppi",
        "pi_prev": "pi_ppi_prev",
        "pi_expect": "Epi_spf_gdp",
    },
    "cpi": {
        "pi": "pi_cpi",
        "pi_prev": "pi_cpi_prev",
        "pi_expect": "Epi_spf_cpi",
    },
}

model_grid = [
    {
        "model_id": "S1_U_G",
        "set": 1,
        "inflation": "ppi",
        "gap": "unemp_gap",
        "N": "N_Gustavo",
    },
    {
        "model_id": "S1_U_T",
        "set": 1,
        "inflation": "ppi",
        "gap": "unemp_gap",
        "N": "N_TNIC",
    },
    {
        "model_id": "S1_Y_G",
        "set": 1,
        "inflation": "ppi",
        "gap": "output_gap_BN",
        "N": "N_Gustavo",
    },
    {
        "model_id": "S1_Y_T",
        "set": 1,
        "inflation": "ppi",
        "gap": "output_gap_BN",
        "N": "N_TNIC",
    },
    {
        "model_id": "S2_U_G",
        "set": 2,
        "inflation": "cpi",
        "gap": "unemp_gap",
        "N": "N_Gustavo",
    },
    {
        "model_id": "S2_U_T",
        "set": 2,
        "inflation": "cpi",
        "gap": "unemp_gap",
        "N": "N_TNIC",
    },
    {
        "model_id": "S2_Y_G",
        "set": 2,
        "inflation": "cpi",
        "gap": "output_gap_BN",
        "N": "N_Gustavo",
    },
    {
        "model_id": "S2_Y_T",
        "set": 2,
        "inflation": "cpi",
        "gap": "output_gap_BN",
        "N": "N_TNIC",
    },
]

## 4. check sample periods

sample_info = []

for spec in model_grid:
    sample = make_model_sample(
        data=data,
        spec=spec,
        inflation_specs=inflation_specs,
        include_N=True,
    )

    sample_info.append({
        "model_id": spec["model_id"],
        "set": spec["set"],
        "inflation": spec["inflation"],
        "gap": spec["gap"],
        "N": spec["N"],
        "start": sample.index.min(),
        "end": sample.index.max(),
        "T": len(sample),
    })

sample_info_df = pd.DataFrame(sample_info)
display(sample_info_df)


,model_id,set,inflation,gap,N,start,end,T
0,S1_U_G,1,ppi,unemp_gap,N_Gustavo,1974-03-31 23:59:59.999999999,2012-12-31 23:59:59.999999999,155
1,S1_U_T,1,ppi,unemp_gap,N_TNIC,1988-03-31 23:59:59.999999999,2022-12-31 23:59:59.999999999,140
2,S1_Y_G,1,ppi,output_gap_BN,N_Gustavo,1974-03-31 23:59:59.999999999,2012-12-31 23:59:59.999999999,155
3,S1_Y_T,1,ppi,output_gap_BN,N_TNIC,1988-03-31 23:59:59.999999999,2022-12-31 23:59:59.999999999,140
4,S2_U_G,2,cpi,unemp_gap,N_Gustavo,1981-09-30 23:59:59.999999999,2012-12-31 23:59:59.999999999,126
5,S2_U_T,2,cpi,unemp_gap,N_TNIC,1988-03-31 23:59:59.999999999,2022-12-31 23:59:59.999999999,140
6,S2_Y_G,2,cpi,output_gap_BN,N_Gustavo,1981-09-30 23:59:59.999999999,2012-12-31 23:59:59.999999999,126
7,S2_Y_T,2,cpi,output_gap_BN,N_TNIC,1988-03-31 23:59:59.999999999,2022-12-31 23:59:59.999999999,140


In [3]:

## 5. prior distributions and MCMC parameters

seed = 0
n_iter = 12000
burn = 4000
thin = 5

prior_specs = {
    "alpha": (0.5, 1),
    "kappa_0": (0.1, 1),
    "delta": (0.1, 1),
    "phi_1": (0.7, 1),
    "rho_1": (0.2, 1),
    "rho_2": (0.2, 1),
    "rho": (0.0, 1),
}

display(pd.DataFrame([
    {"parameter": key, "prior_mean": value[0], "prior_sd": value[1]}
    for key, value in prior_specs.items()
]))


## 6. Gibbs MCMC: HSA steady models

gibbs_module = importlib.reload(gibbs_module)
run_hsa_steady = gibbs_module.run_hsa_steady
draws_to_idata = gibbs_module.draws_to_idata

from types import SimpleNamespace
import xarray as xr

dict_draws = {}
dict_idata = {}

for spec in model_grid:
    sample = make_model_sample(
        data=data,
        spec=spec,
        inflation_specs=inflation_specs,
        include_N=True,
    )

    infl = inflation_specs[spec["inflation"]]
    gap = spec["gap"]
    N_col = spec["N"]

    pi = np.asarray(sample[infl["pi"]], dtype=float)
    pi_prev = np.asarray(sample[infl["pi_prev"]], dtype=float)
    pi_expect = np.asarray(sample[infl["pi_expect"]], dtype=float)

    x = np.asarray(sample[gap], dtype=float)
    x_prev = np.asarray(sample[f"{gap}_prev"], dtype=float)
    N = np.asarray(sample[N_col], dtype=float)

    for orth in (False, True):
        label = "corr" if not orth else "uncorr"
        model_name = f"{spec['model_id']}_HSA_steady_{label}"
        out_path = idat_dir / f"{model_name}.nc"

        if out_path.exists():
            print("Loading", model_name, "| found:", out_path)
            posterior = xr.open_dataset(out_path, group="posterior").load()
            dict_idata[model_name] = SimpleNamespace(posterior=posterior)
            continue

        print(
            "Running",
            model_name,
            "| sample:",
            sample.index.min(),
            "to",
            sample.index.max(),
            "| T =",
            len(sample),
        )

        draws = run_hsa_steady(
            pi=pi,
            pi_prev=pi_prev,
            pi_expect=pi_expect,
            x=x,
            x_prev=x_prev,
            N=N,
            prior_specs=prior_specs,
            n_iter=n_iter,
            burn=burn,
            thin=thin,
            rng=np.random.default_rng(seed),
            orth=orth,
        )

        idata = draws_to_idata(draws)
        dict_draws[model_name] = draws
        dict_idata[model_name] = idata
        save_idata_map({model_name: idata}, idat_dir)
        print("Saved", model_name, "to", out_path)

list(dict_idata.keys())

print(f"Prepared {len(dict_idata)} idata objects from {idat_dir}")


,parameter,prior_mean,prior_sd
0,alpha,0.5,1
1,kappa_0,0.1,1
2,delta,0.1,1
3,phi_1,0.7,1
4,rho_1,0.2,1
5,rho_2,0.2,1
6,rho,0.0,1


Running S1_U_G_HSA_steady_corr | sample: 1974-03-31 23:59:59.999999999 to 2012-12-31 23:59:59.999999999 | T = 155
Saved S1_U_G_HSA_steady_corr to /Users/satoshi/Library/CloudStorage/Dropbox/HSA_NKPC_MCMC/results/idata/S1_U_G_HSA_steady_corr.nc
Running S1_U_G_HSA_steady_uncorr | sample: 1974-03-31 23:59:59.999999999 to 2012-12-31 23:59:59.999999999 | T = 155
Saved S1_U_G_HSA_steady_uncorr to /Users/satoshi/Library/CloudStorage/Dropbox/HSA_NKPC_MCMC/results/idata/S1_U_G_HSA_steady_uncorr.nc
Loading S1_U_T_HSA_steady_corr | found: /Users/satoshi/Library/CloudStorage/Dropbox/HSA_NKPC_MCMC/results/idata/S1_U_T_HSA_steady_corr.nc
Loading S1_U_T_HSA_steady_uncorr | found: /Users/satoshi/Library/CloudStorage/Dropbox/HSA_NKPC_MCMC/results/idata/S1_U_T_HSA_steady_uncorr.nc
Running S1_Y_G_HSA_steady_corr | sample: 1974-03-31 23:59:59.999999999 to 2012-12-31 23:59:59.999999999 | T = 155
Saved S1_Y_G_HSA_steady_corr to /Users/satoshi/Library/CloudStorage/Dropbox/HSA_NKPC_MCMC/results/idata/S1_Y_G_H

In [4]:
## posterior summary
display_hmc_results(
    dict_idata,
    prior_specs,
    models_to_show=list(dict_idata.keys()),
    params=("alpha", "kappa_0", "delta", "phi_1", "rho_1", "rho_2", "rho", "sigma_e", "sigma_zeta", "sigma_u", "sigma_eps"),
    title="Posterior summary for HSA steady models",
)


,model,alpha,kappa_0,delta,phi_1,rho_1,rho_2,rho,sigma_e,sigma_zeta,sigma_u,sigma_eps
0,S1_U_G_HSA_steady_corr,0.8177,-59.3309,1.1777,0.9375,1.7819,-0.7855,0.1023,2.5047,0.6521,0.3317,0.3205
1,S1_U_G_HSA_steady_uncorr,0.8090,-65.4746,1.3841,0.9387,1.7834,-0.7869,0.0000,2.4891,0.6536,0.3318,0.3176
2,S1_U_T_HSA_steady_corr,0.8776,0.3803,-0.5353,0.8505,0.9183,0.0216,0.2752,2.6704,0.9743,0.2539,0.2565
3,S1_U_T_HSA_steady_uncorr,0.8946,0.9620,-2.7723,0.8520,0.9541,-0.0060,0.0000,2.3935,0.9725,0.2583,0.2600
4,S1_Y_G_HSA_steady_corr,0.8451,45.5463,-0.3115,0.8971,1.7884,-0.7923,0.0478,2.4935,0.7069,0.3295,0.3188
5,S1_Y_G_HSA_steady_uncorr,0.8397,53.6275,-0.3892,0.8991,1.7858,-0.7898,0.0000,2.4709,0.7054,0.3288,0.3199
6,S1_Y_T_HSA_steady_corr,0.7631,0.4223,-4.4521,0.7298,1.0209,-0.0468,0.2499,1.8146,0.9780,0.2711,0.2751
7,S1_Y_T_HSA_steady_uncorr,0.7439,0.5979,-4.4269,0.7372,1.0245,-0.0488,0.0000,1.7538,0.9789,0.2709,0.2750
8,S2_U_G_HSA_steady_corr,0.6740,0.0293,0.0039,0.9566,0.8857,0.0861,0.0021,0.6531,0.5930,0.2631,0.2625
9,S2_U_G_HSA_steady_uncorr,0.6807,-0.3531,0.1742,0.9583,0.8459,0.0911,0.0000,0.6600,0.5957,0.2615,0.2620


Posterior summary for HSA steady models
                        model       param       mean      ci_2.5     ci_97.5
0      S1_U_G_HSA_steady_corr       alpha   0.817662    0.729598    0.904288
1      S1_U_G_HSA_steady_corr     kappa_0 -59.330907 -230.407738  166.406717
2      S1_U_G_HSA_steady_corr       delta   1.177681   -2.113963    3.964166
3      S1_U_G_HSA_steady_corr       phi_1   0.937479    0.880363    0.994694
4      S1_U_G_HSA_steady_corr       rho_1   1.781865    1.630034    1.916706
..                        ...         ...        ...         ...         ...
171  S2_Y_T_HSA_steady_uncorr         rho   0.000000    0.000000    0.000000
172  S2_Y_T_HSA_steady_uncorr     sigma_e   0.510292    0.434228    0.593104
173  S2_Y_T_HSA_steady_uncorr  sigma_zeta   0.974966    0.870689    1.091325
174  S2_Y_T_HSA_steady_uncorr     sigma_u   0.497563    0.413311    0.589613
175  S2_Y_T_HSA_steady_uncorr   sigma_eps   0.373734    0.305063    0.460009

[176 rows x 5 columns]
